In [6]:
import os
import glob
import re
import time
import numpy as np
import librosa
import sounddevice as sd
from sklearn.ensemble import RandomForestClassifier

# データセットのパス
DATASET_PATH = r"C:\Users\misak\Desktop\karuta_vosk\kimariji_database"

# マイク録音の設定
SAMPLE_RATE = 16000  # サンプリングレート(Hz)
RECORD_SECONDS = 2.0  # 発声検知後の分析時間（2秒間）
RMS_THRESHOLD = 0.015  # 音声検知の閾値（環境に合わせて調整可能）

def extract_features(y, sr=16000, n_mfcc=13):
    """
    音声波形(y)からMFCC（静的特徴）および1次・2次Delta（動的特徴）を抽出する関数
    """
    target_length = int(sr * RECORD_SECONDS)
    
    # 2秒間に満たない場合はパディング、長い場合はトリミング
    if len(y) < target_length:
        y = np.pad(y, (0, target_length - len(y)), mode='constant')
    else:
        y = y[:target_length]
    
    # 1. MFCC (静的特徴: 音素の特徴)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    # 2. Delta MFCC (1次動的特徴: 子音→母音への変化速度)
    delta_mfcc = librosa.feature.delta(mfcc)
    # 3. Delta-Delta MFCC (2次動的特徴: 変化の加速度)
    delta2_mfcc = librosa.feature.delta(mfcc, order=2)
    
    # 時間軸方向の統計量（平均と標準偏差）
    features = []
    for feat in [mfcc, delta_mfcc, delta2_mfcc]:
        features.append(np.mean(feat, axis=1))
        features.append(np.std(feat, axis=1))
        
    return np.hstack(features)

def trim_by_rms(audio, sr=16000, threshold=RMS_THRESHOLD, duration=RECORD_SECONDS):
    """
    RMS（実効値/音量）を用いて、音声が始まり（発声）の瞬間を検出し、
    そこから指定秒数（2秒間）を切り出す関数
    """
    # フレームごとのRMS（音量エネルギー）を計算
    rms = librosa.feature.rms(y=audio)[0]
    
    # 閾値を超える最初のフレーム（音が出始めた瞬間）を探す
    active_frames = np.where(rms > threshold)[0]
    
    if len(active_frames) == 0:
        # 音量が小さすぎて検知できなかった場合は先頭から切り出し
        start_sample = 0
    else:
        # RMSの1フレームあたりのサンプル数（デフォルト: hop_length=512）
        hop_length = 512
        # 少し手前（約0.05秒前）から取得して子音の立ち上がり（「k」「t」「s」など）の頭切れを防ぐ
        start_frame = max(0, active_frames[0] - 2)
        start_sample = start_frame * hop_length
        
    target_samples = int(sr * duration)
    end_sample = start_sample + target_samples
    
    return audio[start_sample:end_sample]

def load_and_train():
    """
    データセットからファイル名（例: 001_u.wav）を解析し、学習を行う関数
    """
    print("音声データをスキャンして学習中...")
    
    audio_files = []
    for ext in ['*.wav', '*.WAV', '*.mp3', '*.MP3', '*.flac']:
        audio_files.extend(glob.glob(os.path.join(DATASET_PATH, "**", ext), recursive=True))
    
    X = []
    y = []

    for file_path in audio_files:
        filename = os.path.basename(file_path)
        
        # ファイル名から先頭の数字（札番号：例 001）を抽出
        match = re.match(r"^(\d+)", filename)
        if match:
            card_num = match.group(1)
        else:
            print(f"  [警告] 札番号が読み取れないためスキップ: {filename}")
            continue

        try:
            # 余裕をもって多めに読み込んでからRMSで自動トリミング
            audio, sr = librosa.load(file_path, duration=RECORD_SECONDS + 1.0, sr=SAMPLE_RATE)
            trimmed_audio = trim_by_rms(audio, sr=sr)
            
            feat = extract_features(trimmed_audio, sr=sr)
            X.append(feat)
            y.append(card_num)
        except Exception as e:
            print(f"  [エラー] {filename} の読み込み失敗: {e}")

    if len(X) == 0:
        print("\n【エラー】有効な学習用ファイルが見つかりませんでした。")
        return None

    print(f"合計 {len(X)} 件のデータを学習しました！")
    
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X, y)
    
    return clf

def record_and_predict(model):
    """
    音声を待機し、RMSで発声検知してから2秒間を取得・推定する関数
    """
    print("\n" + "="*40)
    input("【Enterキー】を押すとマイク待機モードになります...")
    print("🎙️ 待機中... 声（札の読み）を出してください！")
    
    chunk_duration = 0.1  # 0.1秒ごとにストリーミング監視
    chunk_samples = int(SAMPLE_RATE * chunk_duration)
    buffer = []
    detected = False
    
    # マイクからリアルタイム監視開始
    with sd.InputStream(samplerate=SAMPLE_RATE, channels=1, dtype='float32') as stream:
        while not detected:
            chunk, _ = stream.read(chunk_samples)
            chunk = chunk.flatten()
            
            # リアルタイムRMS（音量）チェック
            rms_val = np.sqrt(np.mean(chunk**2))
            
            if rms_val > RMS_THRESHOLD:
                print("🔊 音声を検知しました！2秒間録音中...")
                buffer.extend(chunk)
                
                # 残りの2秒分（約1.9秒）を追加録音
                remaining_samples = int(SAMPLE_RATE * RECORD_SECONDS) - len(buffer)
                if remaining_samples > 0:
                    remaining_chunk, _ = stream.read(remaining_samples)
                    buffer.extend(remaining_chunk.flatten())
                
                detected = True
            time.sleep(0.01)

    audio_data = np.array(buffer)
    print("分析中...")
    
    # 特徴量抽出と推定
    feat = extract_features(audio_data, sr=SAMPLE_RATE)
    feat = feat.reshape(1, -1)
    
    predicted_card = model.predict(feat)[0]
    probabilities = model.predict_proba(feat)[0]
    max_prob = np.max(probabilities) * 100
    
    print("="*40)
    print(f"🎯 推定結果: 【 札番号 : {predicted_card} 番 】 (確信度: {max_prob:.1f}%)")
    print("="*40)

if __name__ == "__main__":
    clf_model = load_and_train()
    
    if clf_model is not None:
        while True:
            record_and_predict(clf_model)
            
            cont = input("\nもう一度テストしますか？ (y/n): ").strip().lower()
            if cont != 'y':
                print("プログラムを終了します。")
                break

音声データをスキャンして学習中...
合計 200 件のデータを学習しました！

🎙️ 待機中... 声（札の読み）を出してください！
🔊 音声を検知しました！2秒間録音中...
分析中...
🎯 推定結果: 【 札番号 : 015 番 】 (確信度: 11.0%)

🎙️ 待機中... 声（札の読み）を出してください！
🔊 音声を検知しました！2秒間録音中...
分析中...
🎯 推定結果: 【 札番号 : 011 番 】 (確信度: 10.0%)

🎙️ 待機中... 声（札の読み）を出してください！
🔊 音声を検知しました！2秒間録音中...
分析中...
🎯 推定結果: 【 札番号 : 011 番 】 (確信度: 12.0%)
プログラムを終了します。


In [3]:
import os
import glob
import soundfile as sf

FOLDER_PATH = r"C:\Users\misak\Desktop\serino"

def check_audio_specs(folder_path):
    # 対応する拡張子（.ogg, .wav, .flac 等）を取得
    extensions = ['*.ogg', '*.OGG', '*.wav', '*.WAV', '*.flac', '*.mp3']
    audio_files = []
    for ext in extensions:
        audio_files.extend(glob.glob(os.path.join(folder_path, "**", ext), recursive=True))
    
    print(f"対象フォルダ: {folder_path}")
    print(f"発見された音声ファイル: {len(audio_files)} 件\n")
    
    if len(audio_files) == 0:
        print("音声ファイルが見つかりませんでした。フォルダパスを確認してください。")
        return

    # 結果を記録する辞書
    spec_summary = {}

    print(f"{'ファイル名':<35} | {'サンプリングレート':<15} | {'チャンネル数':<10} | {'長さ(秒)':<8}")
    print("-" * 75)

    for file_path in audio_files:
        filename = os.path.basename(file_path)
        try:
            # ヘッダー情報のみを高速取得
            info = sf.info(file_path)
            sr = info.samplerate
            channels = info.channels
            duration = round(info.duration, 2)
            
            # チャンネル表示の補助
            ch_str = f"{channels} (モノラル)" if channels == 1 else f"{channels} (ステレオ)" if channels == 2 else f"{channels} ch"

            # 画面に出力（表示崩れ防止のため長すぎるファイル名は省略）
            display_name = filename if len(filename) <= 33 else filename[:30] + "..."
            print(f"{display_name:<35} | {sr:<7} Hz        | {ch_str:<10} | {duration}s")

            # 集計用
            key = (sr, channels)
            spec_summary[key] = spec_summary.get(key, 0) + 1

        except Exception as e:
            print(f"{filename:<35} | 読み込みエラー: {e}")

    # === 集計結果の表示 ===
    print("\n" + "="*50)
    print("📊 設定の集計結果 (Summary)")
    print("="*50)
    for (sr, ch), count in spec_summary.items():
        ch_name = "モノラル" if ch == 1 else "ステレオ" if ch == 2 else f"{ch}ch"
        print(f"・ サンプリングレート: {sr} Hz / チャンネル: {ch} ({ch_name}) ➔ {count} 件")
    print("="*50)

if __name__ == "__main__":
    check_audio_specs(FOLDER_PATH)

対象フォルダ: C:\Users\misak\Desktop\serino
発見された音声ファイル: 404 件

ファイル名                               | サンプリングレート       | チャンネル数     | 長さ(秒)   
---------------------------------------------------------------------------
芹野恵子_000_1.ogg                      | 44100   Hz        | 1 (モノラル)   | 16.67s
芹野恵子_000_2.ogg                      | 44100   Hz        | 1 (モノラル)   | 8.31s
芹野恵子_001_1.ogg                      | 44100   Hz        | 1 (モノラル)   | 7.14s
芹野恵子_001_2.ogg                      | 44100   Hz        | 1 (モノラル)   | 8.61s
芹野恵子_002_1.ogg                      | 44100   Hz        | 1 (モノラル)   | 6.94s
芹野恵子_002_2.ogg                      | 44100   Hz        | 1 (モノラル)   | 7.09s
芹野恵子_003_1.ogg                      | 44100   Hz        | 1 (モノラル)   | 6.41s
芹野恵子_003_2.ogg                      | 44100   Hz        | 1 (モノラル)   | 7.13s
芹野恵子_004_1.ogg                      | 44100   Hz        | 1 (モノラル)   | 6.99s
芹野恵子_004_2.ogg                      | 44100   Hz        | 1 (モノラル)   | 7.68s
芹野恵子_005_1.ogg   

In [2]:
import os
import subprocess

# 入力元フォルダと出力先フォルダのパス指定
INPUT_DIR = r"C:\Users\misak\Desktop\serino_ogg"
OUTPUT_DIR = r"C:\Users\misak\Desktop\serino"

# 出力先フォルダ（serino）が存在しない場合は新規作成
os.makedirs(OUTPUT_DIR, exist_ok=True)

# フォルダ内のファイルを一括処理
files = [f for f in os.listdir(INPUT_DIR) if f.endswith(".ogg")]
total_files = len(files)

print(f"変換を開始します... (対象ファイル数: {total_files} 件)")

for index, filename in enumerate(files, start=1):
    input_path = os.path.join(INPUT_DIR, filename)
    output_path = os.path.join(OUTPUT_DIR, filename)

    # FFmpegコマンドでモノラル(1ch)・44100Hzに統一変換
    cmd = [
        "ffmpeg",
        "-y",                   # 同名ファイルが存在する場合上書き
        "-i", input_path,        # 入力ファイル
        "-ac", "1",              # チャンネル数を1(モノラル)に統一
        "-ar", "44100",          # サンプリングレート44100Hzに指定
        "-c:a", "libvorbis",      # OGG形式の音声コーデック
        output_path
    ]

    # コマンドを実行（コンソール出力を非表示にしてスッキリ表示）
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    # 進捗の表示
    print(f"[{index}/{total_files}] 変換完了: {filename}")

print("\nすべてのファイルのモノラル変換が完了しました！")
print(f"保存先: {OUTPUT_DIR}")

変換を開始します... (対象ファイル数: 202 件)
[1/202] 変換完了: 芹野恵子_000_1.ogg
[2/202] 変換完了: 芹野恵子_000_2.ogg
[3/202] 変換完了: 芹野恵子_001_1.ogg
[4/202] 変換完了: 芹野恵子_001_2.ogg
[5/202] 変換完了: 芹野恵子_002_1.ogg
[6/202] 変換完了: 芹野恵子_002_2.ogg
[7/202] 変換完了: 芹野恵子_003_1.ogg
[8/202] 変換完了: 芹野恵子_003_2.ogg
[9/202] 変換完了: 芹野恵子_004_1.ogg
[10/202] 変換完了: 芹野恵子_004_2.ogg
[11/202] 変換完了: 芹野恵子_005_1.ogg
[12/202] 変換完了: 芹野恵子_005_2.ogg
[13/202] 変換完了: 芹野恵子_006_1.ogg
[14/202] 変換完了: 芹野恵子_006_2.ogg
[15/202] 変換完了: 芹野恵子_007_1.ogg
[16/202] 変換完了: 芹野恵子_007_2.ogg
[17/202] 変換完了: 芹野恵子_008_1.ogg
[18/202] 変換完了: 芹野恵子_008_2.ogg
[19/202] 変換完了: 芹野恵子_009_1.ogg
[20/202] 変換完了: 芹野恵子_009_2.ogg
[21/202] 変換完了: 芹野恵子_010_1.ogg
[22/202] 変換完了: 芹野恵子_010_2.ogg
[23/202] 変換完了: 芹野恵子_011_1.ogg
[24/202] 変換完了: 芹野恵子_011_2.ogg
[25/202] 変換完了: 芹野恵子_012_1.ogg
[26/202] 変換完了: 芹野恵子_012_2.ogg
[27/202] 変換完了: 芹野恵子_013_1.ogg
[28/202] 変換完了: 芹野恵子_013_2.ogg
[29/202] 変換完了: 芹野恵子_014_1.ogg
[30/202] 変換完了: 芹野恵子_014_2.ogg
[31/202] 変換完了: 芹野恵子_015_1.ogg
[32/202] 変換完了: 芹野恵子_015_2.ogg
[33/202] 変換完了: 芹野恵子_